# Pratica 11 — Regressione logistica e modelli lineari generalizzati

**Corso:** Statistica Applicata con Python · **Blocco 4:** Modelli ed esperimenti · **Tempo stimato:** 60 minuti
**Lezione associata:** `notebooks_clase/clase11_regresion_logistica_glm_IT.ipynb`

## Come funziona questa pratica

- Ogni esercizio ha una cella con `# TODO`: completala.
- Alla fine di ogni esercizio c'è un **controllo automatico** (`assert`). Se fallisce, il messaggio ti indica cosa rivedere. Se dimentichi di completare un `TODO`, il messaggio te lo ricorda.
- Gli esercizi con simulazione o con dati casuali usano tolleranze basate sull'errore standard, quindi un risultato corretto non dovrebbe fallire per caso.
- Le **soluzioni** sono alla fine del notebook, in una sezione a parte. **Non aprirle prima di aver provato.**
- Esegui le celle in ordine. Tutti i dati sono sintetici: *esempi illustrativi*.

---
## Cella 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings

rng = np.random.default_rng(42)          # seme fisso: tutti otteniamo gli stessi risultati
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

# Coefficienti VERI con cui generiamo i dati (li conosciamo solo perché sono sintetici)
VERDAD = {"Intercept": -1.8, "antiguedad": -0.03, "gasto": 0.03, "datos_gb": 0.0, "tickets": 0.45, "plan_anual": -0.9}

def generar_clientes(n, rng):
    """Genera n clienti sintetici (esempio illustrativo) con abbandono secondo VERDAD."""
    antiguedad = np.clip(rng.gamma(2.0, 14, n), 1, 72).round()
    gasto = np.clip(30 + 0.25 * antiguedad + rng.normal(0, 10, n), 10, None).round(1)
    datos_gb = np.clip(0.09 * gasto + rng.normal(0, 0.25, n), 0.1, None).round(2)   # la useremo nella lezione 12
    tickets = rng.poisson(np.exp(0.008 * (gasto - 40)))
    plan_anual = rng.binomial(1, 0.35, n)
    eta = (VERDAD["Intercept"] + VERDAD["antiguedad"] * antiguedad + VERDAD["gasto"] * gasto
           + VERDAD["tickets"] * tickets + VERDAD["plan_anual"] * plan_anual)
    abandona = rng.binomial(1, 1 / (1 + np.exp(-eta)))
    return pd.DataFrame({"abandona": abandona, "antiguedad": antiguedad, "gasto": gasto,
                         "datos_gb": datos_gb, "tickets": tickets, "plan_anual": plan_anual})

print("Librerie e funzioni caricate correttamente.")

## Esercizio 1 — Da probabilità a odds, log-odds e odds ratio

Con l'array `p` di probabilità calcola: `odds` (p / (1 − p)), `logodds` (il suo logaritmo naturale) e `p_de_vuelta` (la probabilità recuperata dai log-odds con la sigmoide). Poi, un coefficiente `beta = 0.69` dà `razon_odds = exp(beta)`: applicalo a un cliente con `p0 = 0.20` (moltiplica i suoi odds per `razon_odds`) e calcola `p_nueva`, la sua nuova probabilità. Infine, `prob_se_duplica` deve essere `True` se `p_nueva` è il doppio di `p0` (approssimativamente) e `False` altrimenti.

In [ ]:
p = np.array([0.05, 0.20, 0.50, 0.90])
beta = 0.69
p0 = 0.20

odds = None              # TODO
logodds = None           # TODO
p_de_vuelta = None       # TODO: la sigmoide 1 / (1 + exp(-logodds))
razon_odds = None        # TODO
p_nueva = None           # TODO: odds di p0 × razon_odds, e da lì la probabilità
prob_se_duplica = None   # TODO

# Controllo automatico
assert all(v is not None for v in (odds, logodds, p_de_vuelta, razon_odds, p_nueva, prob_se_duplica)), "Completa le righe contrassegnate con TODO"
assert np.allclose(odds, p / (1 - p)), "odds = p / (1 - p)"
assert np.allclose(logodds, np.log(p / (1 - p))), "i log-odds sono il logaritmo naturale degli odds"
assert np.allclose(p_de_vuelta, p), "la sigmoide dei log-odds deve restituire p"
assert np.isclose(razon_odds, np.exp(0.69)), "razon_odds = exp(beta)"
assert np.isclose(p_nueva, 1 / 3, atol=0.005), "odds di p0 = 0,25; nuovi odds = 0,25 × razon_odds ≈ 0,5; p = odds / (1 + odds)"
assert prob_se_duplica is False, "gli odds raddoppiano, ma la probabilità passa da 0,20 a 0,333, non a 0,40"
print(f"odds = {np.round(odds, 3)}   p_nueva = {p_nueva:.3f}")
print("Esercizio 1 corretto ✔")

## Esercizio 2 — Adattare, odds ratio e intervallo

Genera 1 500 clienti con `generar_clientes` e adatta con `smf.logit` il modello della variabile `formula` (senza `datos_gb`). Calcola:

- `modelo`: il risultato di `.fit(disp=0)`,
- `odds_ratios`: `np.exp(modelo.params)`,
- `ic_or`: `np.exp(modelo.conf_int())` (DataFrame con due colonne: 0 e 1),
- `or_tickets_10pct`: l'aumento percentuale degli **odds** per ogni ticket, `(OR − 1) × 100`.

In [ ]:
datos = generar_clientes(1500, np.random.default_rng(2026))
formula = "abandona ~ antiguedad + gasto + tickets + plan_anual"

modelo = None             # TODO: smf.logit(formula, data=datos).fit(disp=0)
odds_ratios = None        # TODO
ic_or = None              # TODO
or_tickets_10pct = None   # TODO

# Controllo automatico
assert all(v is not None for v in (modelo, odds_ratios, ic_or, or_tickets_10pct)), "Completa le righe contrassegnate con TODO"
ref = smf.logit(formula, data=datos).fit(disp=0)
assert np.allclose(modelo.params, ref.params), "adatta smf.logit con la formula e i dati indicati"
assert np.allclose(odds_ratios, np.exp(ref.params)), "odds ratio = exp dei coefficienti"
assert np.allclose(ic_or.values, np.exp(ref.conf_int()).values), "applica exp agli estremi dell'intervallo, non all'ampiezza"
assert np.isclose(or_tickets_10pct, (np.exp(ref.params["tickets"]) - 1) * 100), "(exp(coef di tickets) − 1) × 100"
for v in ("tickets", "plan_anual", "antiguedad"):
    assert abs(modelo.params[v] - VERDAD[v]) < 4 * modelo.bse[v], f"il coefficiente stimato di {v} dovrebbe essere vicino al vero"
print(f"Ogni ticket moltiplica gli odds per {odds_ratios['tickets']:.2f} (+{or_tickets_10pct:.0f} %)")
print("Esercizio 2 corretto ✔")

## Esercizio 3 — Probabilità previste ed effetto marginale

Con `datos` (dell'esercizio 2), il modello `modelo3` è già adattato. Ci sono due profili: `perfil_medio` (24 mesi, 40 €, 1 ticket, senza piano annuale) e `perfil_bajo` (72 mesi, 20 €, 0 ticket, con piano annuale). Con la funzione `con_un_ticket_mas` calcola `cambio_medio` e `cambio_bajo`: la variazione di probabilità prevista (in **punti percentuali**, un numero) aggiungendo un ticket a ciascun profilo. Calcola anche `ame_pp`: l'effetto marginale medio di un ticket in più su **tutti** i clienti di `datos`, in punti percentuali. Infine, `mayor_en_medio` è `True` se la variazione del profilo medio è maggiore di quella del profilo basso (lo stesso odds ratio, diverso effetto).

In [ ]:
modelo3 = smf.logit("abandona ~ antiguedad + gasto + tickets + plan_anual", data=datos).fit(disp=0)
perfil_medio = pd.DataFrame({"antiguedad": [24], "gasto": [40], "tickets": [1], "plan_anual": [0]})
perfil_bajo = pd.DataFrame({"antiguedad": [72], "gasto": [20], "tickets": [0], "plan_anual": [1]})

def con_un_ticket_mas(perfil):
    nuevo = perfil.copy()
    nuevo["tickets"] = nuevo["tickets"] + 1
    return nuevo

cambio_medio = None     # TODO: (modelo3.predict(con_un_ticket_mas(perfil_medio)) − modelo3.predict(perfil_medio)) × 100, come numero
cambio_bajo = None      # TODO
ame_pp = None           # TODO: media, su tutte le righe di datos, della variazione di probabilità sommando 1 a tickets
mayor_en_medio = None   # TODO

# Controllo automatico
assert all(v is not None for v in (cambio_medio, cambio_bajo, ame_pp, mayor_en_medio)), "Completa le righe contrassegnate con TODO"
ref_m = float((modelo3.predict(con_un_ticket_mas(perfil_medio)) - modelo3.predict(perfil_medio)).iloc[0] * 100)
ref_b = float((modelo3.predict(con_un_ticket_mas(perfil_bajo)) - modelo3.predict(perfil_bajo)).iloc[0] * 100)
ref_ame = float((modelo3.predict(con_un_ticket_mas(datos)) - modelo3.predict(datos)).mean() * 100)
assert np.isclose(np.squeeze(np.asarray(cambio_medio, dtype=float)), ref_m), "la variazione si misura in punti percentuali: (p dopo − p prima) × 100"
assert np.isclose(np.squeeze(np.asarray(cambio_bajo, dtype=float)), ref_b), "usa perfil_bajo con la funzione con_un_ticket_mas"
assert np.isclose(ame_pp, ref_ame), "l'AME media la variazione di probabilità di OGNI cliente di datos"
assert bool(mayor_en_medio) is True, "confronta cambio_medio con cambio_bajo: la probabilità di partenza bassa lascia poco margine"
print(f"Variazione con +1 ticket: profilo medio {ref_m:.1f} punti, profilo basso {ref_b:.1f} punti, AME {ref_ame:.1f} punti")
print("Esercizio 3 corretto ✔")

## Esercizio 4 — Variabile categoriale: dummy e lettura dell'odds ratio

Con `datos` crea la colonna `tramo` raggruppando `antiguedad` (fino a 11 mesi `"nuevo"`, da 12 a 35 `"medio"`, 36 o più `"veterano"`). Adatta `m_tramo` con `smf.logit`, usando `"nuevo"` come categoria di riferimento e le variabili `gasto`, `tickets` e `plan_anual`. Calcola `or_veterano` (odds ratio di `veterano` rispetto a `nuevo`), `pct_menos_odds` (di quale percentuale riduce gli odds, `(1 − OR) × 100`) e `or_vet_vs_medio` (odds ratio di `veterano` rispetto a `medio`: `exp(b_veterano − b_medio)`).

In [ ]:
datos["tramo"] = pd.cut(datos["antiguedad"], [0, 11, 35, 72], labels=["nuevo", "medio", "veterano"]).astype(str)
f_tramo = "abandona ~ C(tramo, Treatment(reference='nuevo')) + gasto + tickets + plan_anual"
n_vet = "C(tramo, Treatment(reference='nuevo'))[T.veterano]"
n_med = "C(tramo, Treatment(reference='nuevo'))[T.medio]"

m_tramo = None          # TODO: smf.logit(f_tramo, data=datos).fit(disp=0)
or_veterano = None      # TODO: exp del coefficiente n_vet
pct_menos_odds = None   # TODO
or_vet_vs_medio = None  # TODO

# Controllo automatico
assert all(v is not None for v in (m_tramo, or_veterano, pct_menos_odds, or_vet_vs_medio)), "Completa le righe contrassegnate con TODO"
ref = smf.logit(f_tramo, data=datos).fit(disp=0)
assert np.allclose(m_tramo.params, ref.params), "adatta smf.logit con f_tramo e i dati"
assert np.isclose(or_veterano, np.exp(ref.params[n_vet])), "or_veterano = np.exp(m_tramo.params[n_vet])"
assert or_veterano < 1, "i clienti veterani abbandonano meno dei nuovi: l'odds ratio è minore di 1"
assert np.isclose(pct_menos_odds, (1 - np.exp(ref.params[n_vet])) * 100), "(1 − odds ratio) × 100"
assert np.isclose(or_vet_vs_medio, np.exp(ref.params[n_vet] - ref.params[n_med])), "veterano rispetto a medio: exp(b_veterano − b_medio), o il rapporto tra i due odds ratio"
print(f"Odds ratio veterano rispetto a nuevo: {or_veterano:.2f} (−{pct_menos_odds:.0f} % di odds); veterano rispetto a medio: {or_vet_vs_medio:.2f}")
print("Esercizio 4 corretto ✔")

## Esercizio 5 — La famiglia GLM: Binomiale e Poisson

Con `datos` (esercizio 2): adatta con `smf.glm` e famiglia **Binomiale** lo stesso modello dell'esercizio 2 (`glm_bin`) e verifica che i coefficienti coincidano con quelli di `modelo` (`dif_max`: la massima differenza assoluta). Poi adatta una regressione di **Poisson** per `tickets ~ gasto` (`glm_poisson`) e calcola `razon_tasas_10`: per quanto si moltiplica il numero medio di ticket con 10 € in più di spesa, `exp(10 × coefficiente)`.

In [ ]:
glm_bin = None        # TODO: smf.glm(formula, data=datos, family=sm.families.Binomial()).fit()
dif_max = None        # TODO
glm_poisson = None    # TODO: smf.glm("tickets ~ gasto", ..., family=sm.families.Poisson()).fit()
razon_tasas_10 = None # TODO

# Controllo automatico
assert all(v is not None for v in (glm_bin, dif_max, glm_poisson, razon_tasas_10)), "Completa le righe contrassegnate con TODO"
assert dif_max < 1e-6, "un GLM Binomiale con collegamento logit deve dare gli stessi coefficienti di smf.logit"
ref_po = smf.glm("tickets ~ gasto", data=datos, family=sm.families.Poisson()).fit()
assert np.isclose(razon_tasas_10, np.exp(10 * ref_po.params["gasto"])), "exp(10 * coefficiente di gasto) del modello di Poisson"
assert 1.0 < razon_tasas_10 < 1.4, "con 10 € in più di spesa, il numero medio di ticket sale di pochi punti percentuali"
print(f"dif_max = {dif_max:.1e}   rapporto tra tassi per 10 €: {razon_tasas_10:.3f}")
print("Esercizio 5 corretto ✔")

---
## Sfida finale — Adatta e interpreta un modello di abbandono

Con 3 000 clienti nuovi (`datos_r`), adatta il modello logistico di `formula_r` (senza selezione delle variabili né VIF: sono della lezione 12) e calcola: `modelo_r`; `tabla_or` (DataFrame con le colonne `"OR"`, `"IC_inf"` e `"IC_sup"`, una riga per variabile e **senza** l'Intercept); `p_perfil` (probabilità prevista di un cliente di 12 mesi, 45 €, 2 ticket e senza piano annuale, come numero); e `ame_tickets` (effetto marginale medio, in punti percentuali, di un ticket in più). Scrivi in `frases` una lista di 3 o 4 frasi (ciascuna di almeno 40 caratteri) che leggano correttamente gli odds ratio di `tickets`, `plan_anual` e `gasto` (per 10 €): con "le altre variabili fisse", in termini di **odds**, non di probabilità.

In [ ]:
rng_r = np.random.default_rng(99)
datos_r = generar_clientes(3000, rng_r)
formula_r = "abandona ~ antiguedad + gasto + tickets + plan_anual"
perfil_r = pd.DataFrame({"antiguedad": [12], "gasto": [45], "tickets": [2], "plan_anual": [0]})

modelo_r = None     # TODO
tabla_or = None     # TODO: DataFrame con colonne "OR", "IC_inf", "IC_sup" (senza Intercept)
p_perfil = None     # TODO
ame_tickets = None  # TODO
frases = None       # TODO: lista di 3 o 4 stringhe di testo

# Controllo automatico
assert all(v is not None for v in (modelo_r, tabla_or, p_perfil, ame_tickets, frases)), "Completa le righe contrassegnate con TODO"
ref_r = smf.logit(formula_r, data=datos_r).fit(disp=0)
assert np.allclose(modelo_r.params, ref_r.params), "adatta smf.logit con formula_r e datos_r"
ref_ic = np.exp(ref_r.conf_int()).drop(index="Intercept")
assert list(tabla_or.columns) == ["OR", "IC_inf", "IC_sup"] and "Intercept" not in tabla_or.index, "colonne OR, IC_inf, IC_sup e senza la riga Intercept"
assert np.allclose(tabla_or["OR"], np.exp(ref_r.params).drop(index="Intercept")) and np.allclose(tabla_or["IC_inf"], ref_ic[0]) and np.allclose(tabla_or["IC_sup"], ref_ic[1]), "exp dei coefficienti e degli estremi dell'intervallo"
assert np.isclose(np.squeeze(np.asarray(p_perfil, dtype=float)), float(ref_r.predict(perfil_r).iloc[0])), "modelo_r.predict(perfil_r)"
d_mas = datos_r.copy(); d_mas["tickets"] += 1
assert np.isclose(ame_tickets, float((ref_r.predict(d_mas) - ref_r.predict(datos_r)).mean() * 100)), "AME: media della variazione di probabilità di ogni cliente, in punti percentuali"
assert isinstance(frases, list) and 3 <= len(frases) <= 4 and all(isinstance(f, str) and len(f) >= 40 for f in frases), "scrivi 3 o 4 frasi di almeno 40 caratteri ciascuna"
assert sum("odds" in f.lower() for f in frases) >= 2, "leggi l'odds ratio in termini di odds"
print(tabla_or.round(3))
print(f"Probabilità del profilo: {p_perfil:.3f}   AME di un ticket: {ame_tickets:.1f} punti")
print("Sfida finale corretta ✔")

---
## Chiusura della pratica

Hai percorso il ciclo completo di lettura di un modello logistico: convertire tra probabilità, odds e log-odds, adattare e interpretare odds ratio con il loro intervallo, leggere una variabile categoriale rispetto al suo riferimento, calcolare probabilità ed effetti marginali e collocare la logistica nei GLM. Prosegui con la lezione 12: Multicollinearità, selezione delle variabili e validazione.

---
## Soluzioni (non guardare prima di aver provato!)

Ogni cella seguente contiene la soluzione di un esercizio. Puoi eseguirle per confrontarle con il tuo risultato.

### Soluzione dell'esercizio 1

In [ ]:
p = np.array([0.05, 0.20, 0.50, 0.90])
beta = 0.69
p0 = 0.20

odds = p / (1 - p)
logodds = np.log(odds)
p_de_vuelta = 1 / (1 + np.exp(-logodds))
razon_odds = np.exp(beta)
odds_nuevos = (p0 / (1 - p0)) * razon_odds
p_nueva = odds_nuevos / (1 + odds_nuevos)
prob_se_duplica = bool(np.isclose(p_nueva, 2 * p0, atol=0.02))

assert all(v is not None for v in (odds, logodds, p_de_vuelta, razon_odds, p_nueva, prob_se_duplica)), "Completa le righe contrassegnate con TODO"
assert np.allclose(odds, p / (1 - p)), "odds = p / (1 - p)"
assert np.allclose(logodds, np.log(p / (1 - p))), "i log-odds sono il logaritmo naturale degli odds"
assert np.allclose(p_de_vuelta, p), "la sigmoide dei log-odds deve restituire p"
assert np.isclose(razon_odds, np.exp(0.69)), "razon_odds = exp(beta)"
assert np.isclose(p_nueva, 1 / 3, atol=0.005), "odds di p0 = 0,25; nuovi odds = 0,25 × razon_odds ≈ 0,5; p = odds / (1 + odds)"
assert prob_se_duplica is False, "gli odds raddoppiano, ma la probabilità passa da 0,20 a 0,333, non a 0,40"
print(f"odds = {np.round(odds, 3)}   p_nueva = {p_nueva:.3f}")
print("Esercizio 1 corretto ✔")

### Soluzione dell'esercizio 2

In [ ]:
datos = generar_clientes(1500, np.random.default_rng(2026))
formula = "abandona ~ antiguedad + gasto + tickets + plan_anual"

modelo = smf.logit(formula, data=datos).fit(disp=0)
odds_ratios = np.exp(modelo.params)
ic_or = np.exp(modelo.conf_int())
or_tickets_10pct = (odds_ratios["tickets"] - 1) * 100

assert all(v is not None for v in (modelo, odds_ratios, ic_or, or_tickets_10pct)), "Completa le righe contrassegnate con TODO"
ref = smf.logit(formula, data=datos).fit(disp=0)
assert np.allclose(modelo.params, ref.params), "adatta smf.logit con la formula e i dati indicati"
assert np.allclose(odds_ratios, np.exp(ref.params)), "odds ratio = exp dei coefficienti"
assert np.allclose(ic_or.values, np.exp(ref.conf_int()).values), "applica exp agli estremi dell'intervallo, non all'ampiezza"
assert np.isclose(or_tickets_10pct, (np.exp(ref.params["tickets"]) - 1) * 100), "(exp(coef di tickets) − 1) × 100"
for v in ("tickets", "plan_anual", "antiguedad"):
    assert abs(modelo.params[v] - VERDAD[v]) < 4 * modelo.bse[v], f"il coefficiente stimato di {v} dovrebbe essere vicino al vero"
print(f"Ogni ticket moltiplica gli odds per {odds_ratios['tickets']:.2f} (+{or_tickets_10pct:.0f} %)")
print("Esercizio 2 corretto ✔")

### Soluzione dell'esercizio 3

In [ ]:
modelo3 = smf.logit("abandona ~ antiguedad + gasto + tickets + plan_anual", data=datos).fit(disp=0)
perfil_medio = pd.DataFrame({"antiguedad": [24], "gasto": [40], "tickets": [1], "plan_anual": [0]})
perfil_bajo = pd.DataFrame({"antiguedad": [72], "gasto": [20], "tickets": [0], "plan_anual": [1]})

def con_un_ticket_mas(perfil):
    nuevo = perfil.copy()
    nuevo["tickets"] = nuevo["tickets"] + 1
    return nuevo

cambio_medio = float((modelo3.predict(con_un_ticket_mas(perfil_medio)) - modelo3.predict(perfil_medio)).iloc[0] * 100)
cambio_bajo = float((modelo3.predict(con_un_ticket_mas(perfil_bajo)) - modelo3.predict(perfil_bajo)).iloc[0] * 100)
ame_pp = float((modelo3.predict(con_un_ticket_mas(datos)) - modelo3.predict(datos)).mean() * 100)
mayor_en_medio = bool(cambio_medio > cambio_bajo)

assert all(v is not None for v in (cambio_medio, cambio_bajo, ame_pp, mayor_en_medio)), "Completa le righe contrassegnate con TODO"
ref_m = float((modelo3.predict(con_un_ticket_mas(perfil_medio)) - modelo3.predict(perfil_medio)).iloc[0] * 100)
ref_b = float((modelo3.predict(con_un_ticket_mas(perfil_bajo)) - modelo3.predict(perfil_bajo)).iloc[0] * 100)
ref_ame = float((modelo3.predict(con_un_ticket_mas(datos)) - modelo3.predict(datos)).mean() * 100)
assert np.isclose(np.squeeze(np.asarray(cambio_medio, dtype=float)), ref_m), "la variazione si misura in punti percentuali: (p dopo − p prima) × 100"
assert np.isclose(np.squeeze(np.asarray(cambio_bajo, dtype=float)), ref_b), "usa perfil_bajo con la funzione con_un_ticket_mas"
assert np.isclose(ame_pp, ref_ame), "l'AME media la variazione di probabilità di OGNI cliente di datos"
assert bool(mayor_en_medio) is True, "confronta cambio_medio con cambio_bajo: la probabilità di partenza bassa lascia poco margine"
print(f"Variazione con +1 ticket: profilo medio {ref_m:.1f} punti, profilo basso {ref_b:.1f} punti, AME {ref_ame:.1f} punti")
print("Esercizio 3 corretto ✔")

### Soluzione dell'esercizio 4

In [ ]:
datos["tramo"] = pd.cut(datos["antiguedad"], [0, 11, 35, 72], labels=["nuevo", "medio", "veterano"]).astype(str)
f_tramo = "abandona ~ C(tramo, Treatment(reference='nuevo')) + gasto + tickets + plan_anual"
n_vet = "C(tramo, Treatment(reference='nuevo'))[T.veterano]"
n_med = "C(tramo, Treatment(reference='nuevo'))[T.medio]"

m_tramo = smf.logit(f_tramo, data=datos).fit(disp=0)
or_veterano = np.exp(m_tramo.params[n_vet])
pct_menos_odds = (1 - or_veterano) * 100
or_vet_vs_medio = np.exp(m_tramo.params[n_vet] - m_tramo.params[n_med])

assert all(v is not None for v in (m_tramo, or_veterano, pct_menos_odds, or_vet_vs_medio)), "Completa le righe contrassegnate con TODO"
ref = smf.logit(f_tramo, data=datos).fit(disp=0)
assert np.allclose(m_tramo.params, ref.params), "adatta smf.logit con f_tramo e i dati"
assert np.isclose(or_veterano, np.exp(ref.params[n_vet])), "or_veterano = np.exp(m_tramo.params[n_vet])"
assert or_veterano < 1, "i clienti veterani abbandonano meno dei nuovi: l'odds ratio è minore di 1"
assert np.isclose(pct_menos_odds, (1 - np.exp(ref.params[n_vet])) * 100), "(1 − odds ratio) × 100"
assert np.isclose(or_vet_vs_medio, np.exp(ref.params[n_vet] - ref.params[n_med])), "veterano rispetto a medio: exp(b_veterano − b_medio), o il rapporto tra i due odds ratio"
print(f"Odds ratio veterano rispetto a nuevo: {or_veterano:.2f} (−{pct_menos_odds:.0f} % di odds); veterano rispetto a medio: {or_vet_vs_medio:.2f}")
print("Esercizio 4 corretto ✔")

### Soluzione dell'esercizio 5

In [ ]:
glm_bin = smf.glm(formula, data=datos, family=sm.families.Binomial()).fit()
dif_max = float(np.abs(glm_bin.params - modelo.params).max())
glm_poisson = smf.glm("tickets ~ gasto", data=datos, family=sm.families.Poisson()).fit()
razon_tasas_10 = float(np.exp(10 * glm_poisson.params["gasto"]))

assert all(v is not None for v in (glm_bin, dif_max, glm_poisson, razon_tasas_10)), "Completa le righe contrassegnate con TODO"
assert dif_max < 1e-6, "un GLM Binomiale con collegamento logit deve dare gli stessi coefficienti di smf.logit"
ref_po = smf.glm("tickets ~ gasto", data=datos, family=sm.families.Poisson()).fit()
assert np.isclose(razon_tasas_10, np.exp(10 * ref_po.params["gasto"])), "exp(10 * coefficiente di gasto) del modello di Poisson"
assert 1.0 < razon_tasas_10 < 1.4, "con 10 € in più di spesa, il numero medio di ticket sale di pochi punti percentuali"
print(f"dif_max = {dif_max:.1e}   rapporto tra tassi per 10 €: {razon_tasas_10:.3f}")
print("Esercizio 5 corretto ✔")

### Soluzione della sfida finale

In [ ]:
rng_r = np.random.default_rng(99)
datos_r = generar_clientes(3000, rng_r)
formula_r = "abandona ~ antiguedad + gasto + tickets + plan_anual"
perfil_r = pd.DataFrame({"antiguedad": [12], "gasto": [45], "tickets": [2], "plan_anual": [0]})

modelo_r = smf.logit(formula_r, data=datos_r).fit(disp=0)
ic_r = np.exp(modelo_r.conf_int())
tabla_or = pd.DataFrame({"OR": np.exp(modelo_r.params), "IC_inf": ic_r[0], "IC_sup": ic_r[1]}).drop(index="Intercept")
p_perfil = float(modelo_r.predict(perfil_r).iloc[0])
datos_mas = datos_r.copy()
datos_mas["tickets"] = datos_mas["tickets"] + 1
ame_tickets = float((modelo_r.predict(datos_mas) - modelo_r.predict(datos_r)).mean() * 100)
frases = [
    "A parità delle altre variabili, ogni ticket in più moltiplica gli odds di abbandono per l'odds ratio di tickets (maggiore di 1), non la probabilità.",
    "Avere il piano annuale moltiplica gli odds di abbandono per un odds ratio minore di 1: li riduce, a parità delle altre variabili.",
    "Dieci euro in più di spesa mensile moltiplicano gli odds per exp(10 × coefficiente di gasto), un aumento moderato degli odds."]

assert all(v is not None for v in (modelo_r, tabla_or, p_perfil, ame_tickets, frases)), "Completa le righe contrassegnate con TODO"
ref_r = smf.logit(formula_r, data=datos_r).fit(disp=0)
assert np.allclose(modelo_r.params, ref_r.params), "adatta smf.logit con formula_r e datos_r"
ref_ic = np.exp(ref_r.conf_int()).drop(index="Intercept")
assert list(tabla_or.columns) == ["OR", "IC_inf", "IC_sup"] and "Intercept" not in tabla_or.index, "colonne OR, IC_inf, IC_sup e senza la riga Intercept"
assert np.allclose(tabla_or["OR"], np.exp(ref_r.params).drop(index="Intercept")) and np.allclose(tabla_or["IC_inf"], ref_ic[0]) and np.allclose(tabla_or["IC_sup"], ref_ic[1]), "exp dei coefficienti e degli estremi dell'intervallo"
assert np.isclose(np.squeeze(np.asarray(p_perfil, dtype=float)), float(ref_r.predict(perfil_r).iloc[0])), "modelo_r.predict(perfil_r)"
d_mas = datos_r.copy(); d_mas["tickets"] += 1
assert np.isclose(ame_tickets, float((ref_r.predict(d_mas) - ref_r.predict(datos_r)).mean() * 100)), "AME: media della variazione di probabilità di ogni cliente, in punti percentuali"
assert isinstance(frases, list) and 3 <= len(frases) <= 4 and all(isinstance(f, str) and len(f) >= 40 for f in frases), "scrivi 3 o 4 frasi di almeno 40 caratteri ciascuna"
assert sum("odds" in f.lower() for f in frases) >= 2, "leggi l'odds ratio in termini di odds"
print(tabla_or.round(3))
print(f"Probabilità del profilo: {p_perfil:.3f}   AME di un ticket: {ame_tickets:.1f} punti")
print("Sfida finale corretta ✔")

---
## Autovalutazione

Annota in una cella di testo o su carta:

- **Quali esercizi ho risolto senza aiuto?**
- **Quale concetto resta poco chiaro?** (ad esempio: odds ratio rispetto a probabilità, categoria di riferimento, effetto marginale, GLM)
- **Tempo dedicato:** ___ minuti
- **Una domanda che voglio portare alla prossima lezione:** ...